# Week 7 Mini Project 1

## Predicting Employee Attrition Using Classification Models

Dataset path: `data/WA_Fn-UseC_-HR-Employee-Attrition.csv`

## Environment Setup

This project uses the dependency file `requirements.txt`.

Run the next cell inside Jupyter to install the required packages into the active notebook environment.

After that, make sure Jupyter is using the `.venv` kernel.

In [5]:
# Run this once in Jupyter if the notebook shows ModuleNotFoundError.
import sys
print(sys.executable)
%pip install -r requirements.txt

/opt/homebrew/Cellar/jupyterlab/4.2.5/libexec/bin/python
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 33.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 18.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 35.7 MB/s eta 0:00:0000:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━

## 1. Import Libraries

In [8]:
# Show which Python executable the notebook kernel is using.
# If this path is not inside `.venv`, switch the Jupyter kernel before continuing.
import sys
print(sys.executable)

# Suppress non-critical warnings to keep notebook output readable.
import warnings

# Save the final trained model as model.pkl for submission.
import joblib
# Plot charts for attrition distribution, feature analysis, and model evaluation.
import matplotlib.pyplot as plt
# Support numerical operations during preprocessing and analysis.
import numpy as np
# Load and manipulate the IBM HR attrition dataset.
import pandas as pd
# Create cleaner statistical plots for EDA.
import seaborn as sns
# Apply different preprocessing steps to numeric and categorical columns.
from sklearn.compose import ColumnTransformer
# Fill missing values if any appear in the dataset or derived features.
from sklearn.impute import SimpleImputer
# Build a linear baseline classifier for attrition prediction.
from sklearn.linear_model import LogisticRegression
# Measure classification quality beyond accuracy for imbalanced attrition data.
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
# Split data fairly and optionally tune model hyperparameters.
from sklearn.model_selection import GridSearchCV, train_test_split
# Combine preprocessing and modeling into a single reproducible workflow.
from sklearn.pipeline import Pipeline
# Encode text categories and scale numeric features for models like logistic regression and SVM.
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Train a non-linear classifier required by the assignment.
from sklearn.svm import SVC
# Train a tree-based model and inspect feature importance.
from sklearn.tree import DecisionTreeClassifier

# Notebook display settings.
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

/opt/homebrew/Cellar/jupyterlab/4.2.5/libexec/bin/python


### Kernel Setup Note

If you get `ModuleNotFoundError`, do this before continuing:

1. Activate the project virtual environment.
2. Install the required packages from `requirements.txt`.
3. Register the virtual environment as a Jupyter kernel.
4. Switch the notebook kernel to that `.venv` kernel.

Run these commands in the terminal:

```bash
source .venv/bin/activate
pip install -r requirements.txt
python -m ipykernel install --user --name iitm-attrition --display-name "Python (.venv iitm-attrition)"
```

Then in Jupyter, select the kernel named `Python (.venv iitm-attrition)`.

## 2. Load Dataset

In [9]:
# Define the CSV path for the IBM HR attrition dataset used in this assignment.
data_path = "data/WA_Fn-UseC_-HR-Employee-Attrition.csv"

# Load the dataset into a pandas DataFrame for analysis and modeling.
df = pd.read_csv(data_path)

# Inspect basic dataset details before starting EDA.
print(f"Dataset shape: {df.shape}")
print("\nColumn names:")
print(df.columns.tolist())

# Preview the first few rows to confirm the file loaded correctly.
df.head()

Dataset shape: (1470, 35)

Column names:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


## 3. Exploratory Data Analysis

## 4. Data Preprocessing

## 5. Model Building

## 6. Model Evaluation

## 7. Feature Importance and Interpretation

## 8. Final Model Saving

## 9. Conclusions